## Video 3.2.1

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn

import utils

print(f"NumPy:  {np.__version__}")
print(f"PyTorch: {torch.__version__}")

In [ ]:
torch.manual_seed(42)
linear_model = nn.Linear(2, 1)

In [ ]:
def train_model(model, data, epochs=500, learning_rate=0.03):
    features = torch.tensor(data[["x1", "x2"]].values, dtype=torch.float32)
    targets = torch.tensor(data["class"].values, dtype=torch.float32).reshape(-1, 1)

    loss_function = nn.BCEWithLogitsLoss()
    
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    for _ in range(epochs):
        optimizer.zero_grad()
        outputs = model(features)
        loss = loss_function(outputs, targets)
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        predictions = (torch.sigmoid(model(features)) >= 0.5).float()
        accuracy = (predictions == targets).float().mean().item()

    print(f"final loss: {loss.item():.4f}")
    print(f"accuracy:   {accuracy:.1%}")

## Example 2: data that needs a nonlinear boundary

Again, we load and plot the data before choosing a model.

In [ ]:
nonlinear_data = pd.read_csv("data2_nonlinear.csv")
utils.display_points(nonlinear_data, "Dataset 2: no single line can separate these classes")

The classes form an XOR pattern: class 0 occupies the top-left and bottom-right, while class 1 occupies the other two regions. Wherever we place one straight line, some points must remain on the wrong side.

### First attempt: two linear layers, but no activation

It is tempting to expect two layers to solve a more complex problem. However, applying one linear transformation after another is still only a linear transformation:

$$W_2(W_1x + b_1) + b_2 = (W_2W_1)x + (W_2b_1 + b_2)$$

The hidden layer has added parameters, but it has not added a bend to the decision boundary.

In [ ]:
torch.manual_seed(42)
two_linear_layers = nn.Sequential(
    nn.Linear(2, 8),
    nn.Linear(8, 1),
)
print(two_linear_layers)

In [ ]:
train_model(two_linear_layers, nonlinear_data)
utils.display_classifications(two_linear_layers, nonlinear_data, "Two linear layers are still linear")

### Add Non-Linear layers

In [ ]:
# Input values
x = torch.linspace(-5, 5, 200)

# Activation functions
relu = nn.ReLU()(x)
sigmoid = nn.Sigmoid()(x)
tanh = nn.Tanh()(x)
gelu = nn.GELU()(x)



# Create 2x2 subplots
fig, axes = plt.subplots(2, 2, figsize=(10, 7))

plots = [
    (axes[0, 0], relu, "ReLU", "max(0, x)"),
    (axes[0, 1], sigmoid, "Sigmoid", "1 / (1 + exp(-x))"),
    (axes[1, 0], tanh, "Tanh", "tanh(x)"),
    (axes[1, 1], gelu, "Gelu", "gelu(x)"),
]

for ax, y, title, formula in plots:
    ax.plot(x.numpy(), y.numpy())
    ax.set_title(f"{title}\n{formula}")
    ax.set_xlabel("Input (x)")
    ax.set_ylabel("Output")
    ax.grid(True)

plt.tight_layout()
plt.show()


### Second attempt: put ReLU between the layers

In [ ]:
torch.manual_seed(42)
relu_model = nn.Sequential(
    nn.Linear(2, 8),
    nn.ReLU(),
    nn.Linear(8, 1),
)
print(relu_model)

In [ ]:
train_model(relu_model, nonlinear_data, epochs=800)
utils.display_classifications(relu_model, nonlinear_data, "ReLU lets the network learn a nonlinear boundary")

## Assignment: classify the spiral

First, load and inspect a more difficult dataset.

In [ ]:
spiral_data = pd.read_csv("data3_spiral.csv")
utils.display_points(spiral_data, "Assignment: two intertwined spirals")

### Your task

A straight line cannot separate these two spirals. Complete `spiral_model` below, then use the provided training and display calls to test it.

Your model must:

1. Accept the two coordinates `x1` and `x2`.
2. Return one logit for binary classification.
3. Use nonlinear activation functions between linear layers.
4. Reach at least 95% accuracy.

Experiment with the number of layers and the width of each layer. Run the cell only after adding layers to the empty model.

In [ ]:
torch.manual_seed(42)
spiral_model = nn.Sequential(
    # Add your layers here.

)

train_model(spiral_model, spiral_data, epochs=2000, learning_rate=0.02)
utils.display_classifications(spiral_model, spiral_data, "Your spiral classifier")